In [14]:
import json
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from collections import defaultdict
import json


In [15]:
A = ["Measles", "Mumps", "Rubella", "Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib", "Polio", "HPV", "Rotavirus", "PCV"]

V = ["M", "MR", "MMR", "TT", "HepB", "Hib", "IPV", "OPV", "DT", "Td", "DTwP", "DTwP-Hib", "Penta", "Hexa", "HPV", "Rotavirus", "PCV"]

A_v = {
    "M": ["Measles"],
    "MR": ["Measles", "Rubella"],
    "MMR": ["Measles", "Mumps", "Rubella"],
    "TT": ["Tetanus"],
    "HepB": ["Hepatitis_B"],
    "Hib": ["Hib"],
    "IPV": ["Polio"],
    "OPV": ["Polio"],
    "DT": ["Diphtheria", "Tetanus"],
    "Td": ["Diphtheria", "Tetanus"],
    "DTwP": ["Diphtheria", "Tetanus", "Pertussis"],
    "DTwP-Hib": ["Diphtheria", "Tetanus", "Pertussis", "Hib"],
    "Penta": ["Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib"],
    "Hexa": ["Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib", "Polio"],
    "HPV": ["HPV"],
    "Rotavirus": ["Rotavirus"],
    "PCV": ["PCV"]
}

P = [
    "AJ_Vaccines",
    "BB_NCIPD",
    "China_National",
    "Bharat_Biotech",
    "Bilthoven",
    "Biological_E",
    "GSK",
    "Haffkine_Bio",
    "LG_Chem",
    "Merck_Sharp",
    "Panacea_Biotec",
    "PT_Bio",
    "Sanofi",
    "Serum_Institute",
    "Pfizer"
]

P_v = {
    "M": ["Serum_Institute", "PT_Bio"],
    "MR": ["Serum_Institute", "Biological_E"],
    "MMR": ["Serum_Institute", "GSK"],
    "TT": ["Serum_Institute", "PT_Bio", "BB_NCIPD", "Biological_E"],
    "HepB": ["Serum_Institute", "LG_Chem"],
    "Hib": ["Serum_Institute"],
    "IPV": ["LG_Chem", "AJ_Vaccines", "Bilthoven", "Sanofi"],
    "OPV": ["Serum_Institute", "PT_Bio", "GSK", "Sanofi", "Panacea_Biotec", "China_National", "Bharat_Biotech", "Haffkine_Bio"],
    "DT": ["PT_Bio", "BB_NCIPD"],
    "Td": ["Serum_Institute", "PT_Bio", "BB_NCIPD", "Biological_E"],
    "DTwP": ["Serum_Institute", "Biological_E"],
    "DTwP-Hib": ["Serum_Institute"],
    "Penta": ["Serum_Institute", "PT_Bio", "Biological_E", "LG_Chem", "Panacea_Biotec"],
    "Hexa": ["Sanofi"],
    "HPV": ["GSK", "Merck_Sharp", "China_National"],
    "Rotavirus": ["Serum_Institute", "GSK", "Bharat_Biotech"],
    "PCV": ["Serum_Institute", "GSK", "Pfizer"]
}



## Define values

In [16]:
# define constants
beta = 10.0  

tmin = 1
tmax = 10

max_tender_length = 5

unit = 1000

Δ = [i for i in range(1, max_tender_length + 1)]

# Generate time periods
T = [*range(tmin, tmax + 1)]

# Calculate delta values
delta = {t: (1 + 0.03) ** t for t in T}

#  Tender cost
g = {t: 1e8/unit for t in T}

# Cost of expanding capacity for each producer
gamma = {p: 1e8/unit for p in P}  

# Inventory holding cost
h = {v: 0.01 for v in V}  

F_time_set = []

for t in T:
    for tau in T:
        if tau >= t:
            if (tau - t + 1) in Δ:
                F_time_set.append((t, tau))






## Read in necessary data

In [17]:
# import start data
filename = "data/Starting_point.xlsx"
starting_points_file_F = pd.read_excel(filename, sheet_name="F_start")

starting_points_vect_F = [
    (row['Antigen'], (row['Starting'], row['Ending']))
    for _, row in starting_points_file_F.iloc[0:].iterrows()
]

# starting_points_file_I = pd.read_excel(filename, sheet_name="I_start")

# starting_points_vect_I = [
#     (row['Vaccine'], (row['Amount']))
#     for _, row in starting_points_file_I.iloc[1:].iterrows()
# ]

#scenario probabilities
with open('data/scenario_pair_probabilities_new.json', 'r') as f:
    probabilities = json.load(f)

# import results
# Define the path to the file
# file_path = 'social_surplus_base.json'
file_path = 'Phase2_L_results_T_10_delta_5_scen_9_trial_1_inv_1_cap._1_cap.inc._1.json'

# Load the JSON file
with open(file_path, 'r') as file:
    data = json.load(file)

### read in and transform price data to dict

In [18]:

# Load the Excel file to examine its structure
file_path = 'data/Vaccine_price_data.xlsx'
xlsx = pd.ExcelFile(file_path)

# Get all sheet names and skip the first two sheets
sheet_names = xlsx.sheet_names[2:]

# Create a nested dictionary with structure: vaccine[producer][year]
vaccine_dict = {}

for sheet in sheet_names:
    # Read each sheet
    df = pd.read_excel(file_path, sheet_name=sheet)
    
    # Create a nested dictionary for each sheet
    sheet_dict = {}
    for _, row in df.iterrows():
        producer = row['Unnamed: 0'] if 'Unnamed: 0' in row else None
        if producer:
            # Initialize dictionary for each producer
            if producer not in sheet_dict:
                sheet_dict[producer] = {}

            # Populate year data
            for col in df.columns:
                if isinstance(col, int):  # Assuming year columns are integers
                    sheet_dict[producer][col] = row[col]

    # Add sheet's nested dictionary to vaccine dictionary
    vaccine_dict[sheet] = sheet_dict

# Displaying a small portion of the resulting nested dictionary structure
vaccine_dict_sample = {sheet: list(vaccine_dict[sheet].items()) for sheet in vaccine_dict}
modified_dict = {}

for key, value in vaccine_dict_sample.items():
    # Split the key by space and keep only the first part
    new_key = key.split()[0]
    # Add the new key with the original value to the new dictionary
    modified_dict[new_key] = value

# Replace the original dictionary with the modified one
vaccine_price_dict = modified_dict

for vaccine, producers_list in vaccine_price_dict.items():
    # Convert list of tuples to a dictionary
    producers_dict = dict(producers_list)
    # Replace the list with the newly created dictionary
    vaccine_price_dict[vaccine] = producers_dict


## Calculate average price per year per vaccine

In [19]:
# Calculate average cost of vaccine per year
avg_prices_per_period = {}

for vaccine, producers in vaccine_price_dict.items():
    avg_prices_per_period[vaccine] = {}
    
    # Collect prices by time period
    prices_by_time = {}
    for producer, values in producers.items():
        for time, price in values.items():
            # Collecting prices properly, ensuring the value is a number
            if isinstance(price, (int, float)):
                if time not in prices_by_time:
                    prices_by_time[time] = []
                prices_by_time[time].append(price)

    # Calculate average price for each time period
    for time, prices in prices_by_time.items():
        avg_prices_per_period[vaccine][time] = sum(prices) / len(prices) if prices else 0

# avg_prices_per_period

## Calculate Tender costs - g[t] * F[a,t,tau] / delta[t]

In [20]:
f_data = data['F']
# Dictionary to store results in the format F[a][(t, tau)] = 1
f_data_modified = {}

# Function to save results in the specified format F[a][(t, tau)] = 1
def find_ones_in_f_modified(data, path=""):
    if isinstance(data, dict):
        for key, value in data.items():
            find_ones_in_f_modified(value, f"{path}.{key}" if path else key)
    elif isinstance(data, list):
        for index, value in enumerate(data):
            find_ones_in_f_modified(value, f"{path}[{index}]")
    elif data == 1:
        keys = path.split('.')
        a = keys[0]
        t_tau = tuple(map(int, keys[1:]))
        
        if a not in f_data_modified:
            f_data_modified[a] = {}
        
        f_data_modified[a][t_tau] = 1

# Execute the function for "F" data
find_ones_in_f_modified(f_data)

# Remove initial conditions from f_data_modified
for antigen, t_tau in starting_points_vect_F:
    if antigen in f_data_modified and t_tau in f_data_modified[antigen]:
        del f_data_modified[antigen][t_tau]

# Update the summation after removing initial conditions
f_data_summed = {}

# Calculate the sum for each antigen based on the given formula
for antigen, values in f_data_modified.items():
    total_sum = 0
    for (t, tau), f_value in values.items():
        # Applying the formula: g[t] * F[a][(t, tau)] / delta[t]
        total_sum += (g[t] * f_value) / delta[t]
        # total_sum += g[t] * f_value
        
    # Store the summed value for each antigen
    f_data_summed[antigen] = total_sum

# Calculate the overall summation across all antigens
F_OBJ_Value = sum(f_data_summed.values())

# Display the total sum for all antigens
print(f"F Objective cost: {F_OBJ_Value}")


F Objective cost: 1994080.4194829382


## Calculate Capacity Extension Costs - gamma[p] * L[p,t] / delta[t] - WORKING

In [21]:
# Assuming the "L" key contains the data you mentioned
L_data = data.get("L", {})

transformed_L_data = {}

# Iterate through each producer in L_data
for producer, years in L_data.items():
    for year, value in years.items():
        # If the year doesn't exist in transformed_L_data, initialize it as an empty dictionary
        if year not in transformed_L_data:
            transformed_L_data[year] = {}
        
        # Set the value for the producer in the corresponding year
        transformed_L_data[year][producer] = value

# The transformed_L_data now has the structure year[producer][value]
# print(transformed_L_data)


In [22]:
# Create a new dictionary to store the results after multiplication
result_after_gamma = {}

# Iterate through each year in transformed_L_data
for year, producers in transformed_L_data.items():
    result_after_gamma[year] = {}
    
    # Iterate through each producer in the year
    for producer, value in producers.items():
        # Multiply the producer's value by the corresponding value from gamma
        result_after_gamma[year][producer] = value * gamma[producer] / delta[int(year)]

# Initialize the total sum variable
L_OBJ_Value = 0

# Iterate through each year in result_after_gamma
for year, producers in result_after_gamma.items():
    # Iterate through each producer in the year and add its value to the total sum
    for producer, value in producers.items():
        L_OBJ_Value += value

# The total_sum variable will contain the sum of all the values
print(f"L Objective Value: {L_OBJ_Value}")



L Objective Value: 10740231.834678644


## Calculate missed doses by scenario - Beta * S[a,t,omega] / delta[t] - WORKING

In [23]:
def process_scenarios(S_data, beta, delta):
    # Step 1: Reorganize data with scenario at the top level
    S_data_by_scenario = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
    for antigen, year_data in S_data.items():
        for year, scenario_data in {y: d for y, d in year_data.items() if y != '0'}.items():
            for scenario, value in scenario_data.items():
                S_data_by_scenario[scenario][year][antigen] = value

    # Convert to regular dictionary
    S_data_by_scenario = dict(S_data_by_scenario)

    # Step 2: Scale data
    S_data_by_scenario_scaled = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
    for scenario, years in S_data_by_scenario.items():
        for year, antigens in years.items():
            for antigen in antigens.keys():
                S_data_by_scenario_scaled[scenario][year][antigen] = (
                    S_data_by_scenario[scenario][year][antigen] * beta / delta[int(year)]
                )

    # Step 3: Calculate scenario sums
    scenario_sums = {}
    for scenario, years in S_data_by_scenario_scaled.items():
        scenario_sum = sum(
            value
            for year in years.values()
            for value in year.values()
            if isinstance(value, (int, float))
        )
        scenario_sums[scenario] = scenario_sum

    return scenario_sums

In [24]:
S_data = data['S']
scenario_sums_S = process_scenarios(S_data, beta, delta)

In [25]:
selected = list(scenario_sums_S.keys())
normalized = {k: probabilities[str(k)] / sum(probabilities[str(i)] for i in selected) for k in selected}


In [26]:
S_OBJ_Values = {k: normalized[k] * scenario_sums_S[k] for k in normalized}
S_OBJ_Value = sum(S_OBJ_Values.values())
# S_OBJ_Value
print(f"Missed Dose OBJ Value: {(S_OBJ_Value)}")

Missed Dose OBJ Value: 676201492.2244489


## Calculate doses purchased - r[v,p,t] * X[v,p,t,omega] / delta[t] 

In [36]:
X_data = data['X']
X_data_by_scenario = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))

# Iterate through the original structure to rearrange the keys with the correct nesting
for vaccine, producers in X_data.items():
    for producer, years in producers.items():
        for year, scenarios in years.items():
            for scenario, value in scenarios.items():
                # Move scenario to the top level, followed by year, vaccine, and then producer
                X_data_by_scenario[scenario][year][vaccine][producer] = value

# Convert to a regular dictionary for easy use
X_data_by_scenario = dict(X_data_by_scenario)

In [37]:
X_data_by_scenario_scaled = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))

for scenario, year_data in X_data_by_scenario.items():
    for years, vaccine_data in year_data.items():
        for vaccines, producer_data in vaccine_data.items():
            for producer, purchases in producer_data.items():
                # print(f"vaccine: {vaccines}, producer: {producer}, year: {years}, scenario: {scenario} purchase: {purchases} ")
                # print(f"Initial Value: {X_data_by_scenario[scenario][year][vaccines][producer]}")
                # print(f"Vaccine cost: {vaccine_price_dict[vaccines][producer][int(year)]}")
                X_data_by_scenario_scaled[scenario][year][vaccines][producer] = X_data_by_scenario[scenario][year][vaccines][producer] * vaccine_price_dict[vaccines][producer][int(year)] / delta[int(year)]


In [38]:
def calculate_scenario_sums(data):
    vax_scenario_sums = {}
    for scenario, years in data.items():
        total_sum = 0
        for year, producers in years.items():
            for vaccine, producers_data in producers.items():
                total_sum += sum(producers_data.values())
        vax_scenario_sums[scenario] = total_sum
    return vax_scenario_sums

# Calculate and print the scenario sums
vax_scenario_sums = calculate_scenario_sums(X_data_by_scenario_scaled)


In [ ]:
X_OBJ_Values = {k: probabilities[k] * vax_scenario_sums[k] for k in probabilities}
X_OBJ_Value = sum(X_OBJ_Values.values())
X_OBJ_Value
print(f"Vaccines Purchased OBJ Value: {X_OBJ_Value}")

## calculate inventory holding costs -h[v] * r_bar[v,t] * I[v,t,omega] / delta[t] 

In [40]:
I_data = data.get("I", {})

reversed_data = defaultdict(lambda: defaultdict(dict))
for vaccine, years in I_data.items():
    for year, scenarios in years.items():
            for scenario, value in scenarios.items():
                reversed_data[scenario][year][vaccine] = value 

In [41]:
def remove_year_zero(data):
    for scenario, years in list(data.items()):
        if isinstance(years, defaultdict):  # Ensure it's a defaultdict or dict
            for year in list(years.keys()):
                if year == '0':  # Check if the year is 0 (string form)
                    del years[year]  # Remove the entry
    return data

reversed_remove_start_I = remove_year_zero(reversed_data)

In [42]:
result = defaultdict(lambda: defaultdict(lambda: defaultdict(float)))

for scenario, years in reversed_remove_start_I.items():
    for year, vaccines in years.items():
        for vaccine, value in vaccines.items():
                result[scenario][year][vaccine] = (value * avg_prices_per_period[vaccine][int(year)] * h[vaccine]) / delta[int(year)]

# The "result" dictionary will contain the multiplied values.


In [ ]:
scenario_sums_H = {}

# Iterate through the `result` dictionary to calculate the sum by scenario.
for scenario, years in result.items():
    scenario_sum = 0  # Initialize the sum for the scenario.
    for year, vaccines in years.items():
        for vaccine, value in vaccines.items():
            scenario_sum += value  # Add the value to the sum for the scenario.
    
    scenario_sums_H[scenario] = scenario_sum  # Store the calculated sum for each scenario.

# `scenario_sums` will contain the total sum for each scenario.
print(scenario_sums_H)

In [ ]:
I_OBJ_Values = {k: probabilities[k] * scenario_sums_H[k] for k in probabilities}
I_OBJ_Value = sum(I_OBJ_Values.values())
print(f"Inventory Holding OBJ Value: {(I_OBJ_Value)}")

# TOTAL OBJ VALUE

In [ ]:
print(f"OBJ Value: {F_OBJ_Value + L_OBJ_Value + X_OBJ_Value + S_OBJ_Value + I_OBJ_Value}")

In [ ]:
print(f"OBJ Value: {S_OBJ_Value}")